### Setup

In [1]:
import pandas as pd
import os
from typhoon_ocr import ocr_document
from bs4 import BeautifulSoup
import re
from rapidfuzz import fuzz, process
from tqdm import tqdm

In [18]:
os.environ['TYPHOON_OCR_API_KEY'] = 'sk-5THr17CaB2d0DtEFJNOA7EDpbC5tF73RAItZ9PI0J8DOkABj'

In [2]:
TEMPLATE_OLD_PATH = "../data/submission_template.csv"
TEMPLATE_PATH = "../data/submission_template_v3.csv"

In [3]:
template_df = pd.read_csv(TEMPLATE_PATH)
template_old_df = pd.read_csv(TEMPLATE_OLD_PATH)
template_df['doc_id'] = template_old_df['doc_id']
template_df

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1
...,...,...,...,...
10048,party_list_34_11_53,ไทยพิทักษ์ธรรม,0,party_list_34_11
10049,party_list_34_11_54,ความหวังใหม่,0,party_list_34_11
10050,party_list_34_11_55,ไทยรวมไทย,0,party_list_34_11
10051,party_list_34_11_56,เพื่อบ้านเมือง,0,party_list_34_11


### EDA

In [4]:
template_df['party_name'].unique()

array(['ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
       'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
       'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
       'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
       'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
       'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
       'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
       'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
       'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ', nan,
       'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
       'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
       'สร้างชาติ', 'ใหม่', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
       'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
       'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
       'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธ

In [5]:
template_df.head()

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1


In [6]:
NaN_df = template_df[
    template_df['party_name'].isna()
]

NaN_df

,id,party_name,votes,doc_id
737,constituency_14_2_10,NaN,0,constituency_14_2
738,constituency_14_2_11,NaN,0,constituency_14_2
739,constituency_14_2_12,NaN,0,constituency_14_2
740,constituency_14_2_13,NaN,0,constituency_14_2
741,constituency_14_2_14,NaN,0,constituency_14_2
742,constituency_14_2_15,NaN,0,constituency_14_2
743,constituency_14_2_16,NaN,0,constituency_14_2
744,constituency_14_2_17,NaN,0,constituency_14_2


In [7]:
# 737 - 744 Correctly Null
template_df.iloc[737]

id            constituency_14_2_10
party_name                     NaN
votes                            0
doc_id           constituency_14_2
Name: 737, dtype: object

### Extraction

In [8]:
def thai_num_to_int(text: str) -> int:

    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # remove non-numeric prefixes
    text = re.sub(r"[^\d,]", " ", text)

    match = re.search(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    return int(match.group().replace(",", ""))

In [9]:
def extract_party_score_dict(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    table = soup.find("table")
    if table is None:
        raise ValueError("No <table> found")

    rows = table.find_all("tr")
    if not rows:
        return {}

    headers = [td.get_text(strip=True) for td in rows[0].find_all(["td", "th"])]

    # Candidate labels
    party_candidates = ["พรรคการเมือง", "ชื่อพรรคการเมือง", "สังกัดพรรคการเมือง"]
    score_candidates = ["ได้คะแนน"]

    party_idx = None
    score_idx = None
    best_party_score = 0
    best_score_score = 0

    for i, h in enumerate(headers):
        # compute best similarity across all candidates
        party_sim = max(fuzz.partial_ratio(h, c) for c in party_candidates)
        score_sim = max(fuzz.partial_ratio(h, c) for c in score_candidates)

        if party_sim > best_party_score:
            best_party_score = party_sim
            party_idx = i

        if score_sim > best_score_score:
            best_score_score = score_sim
            score_idx = i

    if best_party_score < 60 or best_score_score < 60:
        raise ValueError("Required columns not confidently found")

    result = {}

    for row in rows[1:]:
        cols = [td.get_text(strip=True) for td in row.find_all("td")]

        if len(cols) <= max(party_idx, score_idx):
            continue

        key = cols[party_idx]
        value = cols[score_idx]

        try:
            value = thai_num_to_int(value)
        except Exception:
            continue  # safer: skip invalid rows

        result[key] = value

    return result

In [10]:
def extraction(path):
    try:
        if not os.path.exists(path):
            return {}
        markdown = ocr_document(
            pdf_or_image_path=path
        )
        party_dict = extract_party_score_dict(markdown)
        return party_dict
    except Exception as e:
        print(e.__str__())
        return {}

### Inference

In [11]:
def assign_votes(df, vote_dict, threshold=80):
    # Remove non-party keys
    vote_dict = {
        k: v for k, v in vote_dict.items()
        if k != 'รวมคะแนนทั้งสิ้น'
    }

    keys = list(vote_dict.keys())

    def get_vote(party_name):
        match = process.extractOne(
            party_name,
            keys,
            scorer=fuzz.ratio
        )
        
        if match is None:
            return 0
        
        best_key, score, _ = match
        
        if score >= threshold:
            return vote_dict[best_key]
        return 0

    df['votes'] = df['party_name'].apply(get_vote)
    return df

In [12]:
submission_df = template_df.copy()

In [13]:
PREFIX = "../pdf/"

doc_ids = submission_df['doc_id'].unique()

In [ ]:
for doc_id in tqdm(doc_ids, desc="Processing", unit="doc"):
	try:
		vote_dict = extraction(PREFIX + doc_id + '.pdf')

		mask = submission_df['doc_id'] == doc_id

		submission_df.loc[mask] = assign_votes(
			submission_df.loc[mask].copy(),
			vote_dict,
			threshold=80
		)

	except Exception as e:
		print(f"{doc_id}: {str(e)}")
	break

TabError: inconsistent use of tabs and spaces in indentation (3782303161.py, line 15)

In [15]:
submission_df

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1
...,...,...,...,...
10048,party_list_34_11_53,ไทยพิทักษ์ธรรม,0,party_list_34_11
10049,party_list_34_11_54,ความหวังใหม่,0,party_list_34_11
10050,party_list_34_11_55,ไทยรวมไทย,0,party_list_34_11
10051,party_list_34_11_56,เพื่อบ้านเมือง,0,party_list_34_11


In [16]:
output_df = submission_df.drop(['doc_id', 'party_name'], axis=1)

In [17]:
output_df.to_csv("submission2.csv", index=False)